# 06.5 · Exercises

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/06-unsupervised/05-dbscan-gmm/exercises.ipynb)

In [ ]:
# --- data helper: works locally and on Colab -------------------------------
from pathlib import Path
import urllib.request

def data_path(name):
    """Return a local path to data/<name>, downloading from GitHub on Colab."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data" / name
        if candidate.exists():
            return candidate
    url = f"https://raw.githubusercontent.com/cloudops-one/training-ai-ml/main/data/{name}"
    dest = Path(name)
    if not dest.exists():
        urllib.request.urlretrieve(url, dest)
    return dest

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")

In [ ]:
names = list("PQRSTUVWXY")
points = np.array([[1, 1], [1, 2], [2, 1], [2, 2], [3, 2],
                   [6, 6], [6, 7], [7, 6], [8, 7], [4, 8]], dtype=float)

In [ ]:
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import cdist

def dbscan_table(points, eps, min_samples):
    """Neighbour counts and point kinds, next to DBSCAN's labels."""
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    core = np.zeros(len(points), bool); core[db.core_sample_indices_] = True
    near = cdist(points, points) <= eps
    kind = np.where(core, "core", np.where(db.labels_ == -1, "noise", "border"))
    return pd.DataFrame({"neighbours": [", ".join(np.array(names)[row]) for row in near],
                         "count": near.sum(axis=1), "kind": kind, "cluster": db.labels_}, index=names)

## Exercise 1: stricter crowds

By hand, re-run DBSCAN on the ten points with **min_samples = 4** (eps still 1.5). Which points change kind? How many clusters? Check with `dbscan_table(points, 1.5, 4)`.

In [ ]:
# your code here

## Exercise 2: £35

In the GMM example (group 1: mean 20, sd 5; group 2: mean 50, sd 10; equal weights), compute $P(\text{group 1} \mid 35)$ by hand, then in code.

In [ ]:
# your code here

## Exercise 3: eps on the moons

Run DBSCAN (min_samples = 5) on the moons with eps = 0.1, 0.2, 0.3 and 0.5. Plot each. What happens as eps grows?

In [ ]:
from sklearn.datasets import make_moons
moons, moon_truth = make_moons(n_samples=300, noise=0.06, random_state=42)

In [ ]:
# your code here

## Exercise 4: the in-between customers

Fit the 4-component Gaussian mixture on the ShopWave RFM table (as in the lesson). List the 10 customers with the lowest top probability, with their R, F, M. Why is the model unsure?

In [ ]:
import sys
sys.path.insert(0, str(data_path("download.py").parent))
from download import load_retail

raw = load_retail("Year 2010-2011")
sales = raw[~raw["Invoice"].astype(str).str.startswith("C")]
sales = sales[(sales["Quantity"] > 0) & (sales["Price"] > 0)].dropna(subset=["Customer ID"]).drop_duplicates()
sales = sales.assign(Revenue=sales["Quantity"] * sales["Price"])

snapshot = sales["InvoiceDate"].max().normalize() + pd.Timedelta(days=1)    # 2011-12-10
rfm = sales.groupby("Customer ID").agg(
    recency=("InvoiceDate", lambda d: (snapshot - d.max()).days),
    frequency=("Invoice", "nunique"),
    monetary=("Revenue", "sum"))
print(f"{len(sales):,} rows, {len(rfm):,} customers, snapshot {snapshot.date()}")

In [ ]:
from sklearn.preprocessing import StandardScaler

rfm_log = np.log1p(rfm)                         # tame the long right tails
X = StandardScaler().fit_transform(rfm_log)      # mean 0, std 1 per column
print("X shape:", X.shape)

In [ ]:
# your code here